In [1]:
import os
os.chdir('/home/hrayrh/projects/incontext_spurious')
os.environ['DATA_ROOT_DIR'] = '/home/hrayrh/data'

In [2]:
import sys
sys.path.append('/home/hrayrh/projects/incontext_spurious')

In [10]:
from collections import Counter
from matplotlib import pyplot as plt
%matplotlib inline
import numpy as np
from sklearn.linear_model import LogisticRegression
from src.datamodules.datasets import WaterbirdsExtracted

# Inspect extracted files

In [4]:
saved_data = np.load('/home/hrayrh/data/waterbirds/resnet50/test/combined.npz')
saved_data.keys()

KeysView(NpzFile '/home/hrayrh/data/waterbirds/resnet50/test/combined.npz' with keys: encodings, indices_map)

In [5]:
saved_data['encodings'].shape

(5794, 2048)

In [6]:
len(saved_data['indices_map'])

11788

In [7]:
saved_data['indices_map'][:100]

array([ 0, -1,  1, -1, -1,  2, -1, -1, -1,  3, -1,  4,  5,  6, -1,  7,  8,
       -1,  9, -1, -1, 10, -1, 11, 12, 13, 14, 15, 16, -1, -1, -1, 17, 18,
       -1, -1, -1, 19, 20, -1, -1, -1, -1, 21, 22, -1, 23, 24, -1, 25, -1,
       26, -1, 27, -1, -1, 28, -1, 29, -1, 30, -1, 31, -1, 32, -1, -1, 33,
       -1, 34, 35, -1, -1, -1, -1, 36, -1, -1, 37, -1, -1, 38, 39, -1, 40,
       -1, 41, 42, -1, 43, 44, 45, 46, 47, 48, -1, -1, 49, -1, 50])

In [8]:
saved_data['indices_map'][-100:]

array([5746,   -1,   -1, 5747, 5748,   -1, 5749,   -1,   -1,   -1, 5750,
       5751, 5752,   -1,   -1, 5753,   -1,   -1, 5754,   -1, 5755,   -1,
       5756, 5757, 5758, 5759,   -1,   -1,   -1,   -1,   -1, 5760,   -1,
         -1, 5761,   -1, 5762,   -1, 5763,   -1, 5764, 5765,   -1,   -1,
       5766,   -1, 5767,   -1, 5768,   -1,   -1, 5769, 5770, 5771,   -1,
       5772, 5773,   -1, 5774,   -1, 5775,   -1, 5776,   -1,   -1,   -1,
         -1, 5777, 5778,   -1,   -1,   -1, 5779,   -1,   -1, 5780,   -1,
       5781,   -1, 5782, 5783,   -1, 5784,   -1, 5785,   -1, 5786,   -1,
       5787,   -1, 5788, 5789,   -1,   -1, 5790,   -1, 5791, 5792,   -1,
       5793])

# Inspecting WaterbirdsExtracted to see how well Y and S are predictable from encodings


**Resnet-50**

512 examples
* Y acc = 85.07
* S acc = 81.31

full dataset (4k examples)
* Y acc@full = 81.81
* S acc@full = 79.89


**Dino V2**

512 examples
* Y acc = 94.49
* S acc = 65.14

full dataset (4k examples)
* Y acc@full = 94.24
* S acc@full = 79.81

In [29]:
ds = WaterbirdsExtracted(
    root_dir='/home/hrayrh/data',
    encoding_extractor='dinov2_vitb14',
    # encoding_extractor='resnet50',
)

In [30]:
train = ds.get_subset('train')
val = ds.get_subset('val')
test = ds.get_subset('test')
print(f'{len(train)=}, {len(val)=}, {len(test)=}')

len(train)=4795, len(val)=1199, len(test)=5794


In [31]:
def compute_group_proportions(examples):
    g = 2 * examples[:, 2] + examples[:, 1]  # 2 * y + c
    return [np.mean(g==i) for i in range(4)]

In [32]:
train_X, examples = train[np.arange(len(train))]
print(f'{train_X.shape=}, {examples.shape=}')
train_s = examples[:, 1]
train_y = examples[:, 2]
print(f'{train_s.shape=}, {train_y.shape=}')
print(compute_group_proportions(examples))

train_X.shape=(4795, 768), examples.shape=(4795, 3)
train_s.shape=(4795,), train_y.shape=(4795,)
[0.7295099061522419, 0.0383733055265902, 0.01167883211678832, 0.22043795620437956]


In [33]:
val_X, examples = val[np.arange(len(val))]
print(f'{val_X.shape=}, {examples.shape=}')
val_s = examples[:, 1]
val_y = examples[:, 2]
print(f'{val_s.shape=}, {val_y.shape=}')
print(compute_group_proportions(examples))

val_X.shape=(1199, 768), examples.shape=(1199, 3)
val_s.shape=(1199,), val_y.shape=(1199,)
[0.3894912427022519, 0.3886572143452877, 0.11092577147623019, 0.11092577147623019]


In [34]:
test_X, examples = test[np.arange(len(test))]
print(f'{test_X.shape=}, {examples.shape=}')
test_s = examples[:, 1]
test_y = examples[:, 2]
print(f'{test_s.shape=}, {test_y.shape=}')
print(compute_group_proportions(examples))

test_X.shape=(5794, 768), examples.shape=(5794, 3)
test_s.shape=(5794,), test_y.shape=(5794,)
[0.38919571971004485, 0.38919571971004485, 0.11080428028995512, 0.11080428028995512]


In [35]:
examples[:20]

array([[ 0,  1,  1],
       [ 1,  0,  1],
       [ 2,  1,  1],
       [ 3,  1,  1],
       [ 4,  0,  1],
       [ 5,  0,  1],
       [ 6,  0,  1],
       [ 7,  0,  1],
       [ 8,  1,  1],
       [ 9,  1,  1],
       [10,  0,  1],
       [11,  1,  1],
       [12,  1,  1],
       [13,  1,  1],
       [14,  1,  1],
       [15,  0,  1],
       [16,  0,  1],
       [17,  1,  1],
       [18,  0,  1],
       [19,  1,  1]])

## Predicting Y

In [39]:
log_reg = LogisticRegression(max_iter=10000)
compute_group_proportions(examples)

# selected_indices = np.random.choice(np.arange(len(train_X)), 512, replace=False)
# log_reg.fit(train_X[selected_indices], train_y[selected_indices])

log_reg.fit(train_X, train_y)

val_pred = log_reg.predict(val_X)
val_acc = np.mean(val_pred == val_y)
print(f'predicting Y, {val_acc=}')

predicting Y, val_acc=0.9424520433694745


In [40]:
train_X.shape

(4795, 768)

## Predicting S

In [41]:
log_reg = LogisticRegression(max_iter=10000)

# selected_indices = np.random.choice(np.arange(len(train_X)), 512, replace=False)
# log_reg.fit(train_X[selected_indices], train_s[selected_indices])

log_reg.fit(train_X, train_s)

val_pred = log_reg.predict(val_X)
val_acc = np.mean(val_pred == val_s)
print(f'predicting S, {val_acc=}') 

predicting S, val_acc=0.7981651376146789


## Imbalanced contexts

In [42]:
def worst_group_acc(pred, y, s):
    g = 2 * y + s
    accs = []
    for g_idx in range(4):
        mask = (g == g_idx)
        g_acc = np.mean(pred[mask] == y[mask])
        accs.append(g_acc)
    print('group accuracies:', accs)
    print('overall accuracy:', np.mean(pred==y))
    return np.min(accs)

In [43]:
train_X, examples = train[np.arange(len(train))]
train_s = examples[:, 1]
train_y = examples[:, 2]
g = 2 * examples[:, 2] + examples[:, 1]  # 2 * y + c

target_group_proportions = [0.45, 0.05, 0.05, 0.45]
current_group_proportions = compute_group_proportions(examples)
current_group_counts = len(train) * np.array(current_group_proportions)
target_group_counts = len(train) * np.array(target_group_proportions)
c = np.min(current_group_counts / target_group_counts)
actual_group_counts = c * target_group_counts / 7  # this results in about 1000 total examples, same as for 512 context_class_size

In [44]:
print(current_group_counts)
print(target_group_counts)
print(actual_group_counts)

[3498.  184.   56. 1057.]
[2157.75  239.75  239.75 2157.75]
[72.  8.  8. 72.]


In [45]:
train_mask = np.zeros(len(train), dtype=np.bool_)
g = 2 * examples[:, 2] + examples[:, 1]  # 2 * y + c

for i in range(4):
    g_indices = np.where(g == i)[0]
    selected_g_indices = np.random.choice(
        g_indices,
        int(actual_group_counts[i]),
        replace=False
    )
    train_mask[selected_g_indices] = True

print(np.sum(train_mask))
print(np.sum(actual_group_counts))

160
160.0


In [46]:
# normal ERM
log_reg = LogisticRegression(max_iter=1000)    
log_reg.fit(train_X[train_mask], train_y[train_mask])

# downsampled ERM
if False:
    log_reg = LogisticRegression(max_iter=1000)    
    X = train_X[train_mask]
    y = train_y[train_mask]
    s = train_s[train_mask]
    g = 2 * y + s
    counts = Counter(g)
    print(counts)
    min_count = np.min([counts[g_idx] for g_idx in counts])
    selected_indices = []
    for g_idx in range(4):
        selected_indices.append(
            np.random.choice(np.where(g==g_idx)[0], size=min_count, replace=False)
        )
    selected_indices=np.concatenate(selected_indices, axis=0)
    log_reg.fit(X[selected_indices], y[selected_indices])

In [47]:
val_pred = log_reg.predict(val_X)
print(f'predicting Y, matching Y, wg acc: {worst_group_acc(val_pred, val_y, val_s)}')

group accuracies: [0.9700214132762313, 0.851931330472103, 0.9022556390977443, 0.9548872180451128]
overall accuracy: 0.9149291075896581
predicting Y, matching Y, wg acc: 0.851931330472103


In [48]:
val_pred = log_reg.predict(val_X)
print(f'predicting Y, matching S, wg acc: {worst_group_acc(val_pred, val_s, val_y)}')

group accuracies: [0.9700214132762313, 0.09774436090225563, 0.148068669527897, 0.9548872180451128]
overall accuracy: 0.5521267723102585
predicting Y, matching S, wg acc: 0.09774436090225563
